# ByteCore Phase 19 — phone-size research experiment

**Run all cells in a fresh free Colab T4 runtime** (Runtime → Change runtime type → T4 GPU). This is a standalone notebook; no repository clone is required. It embeds its source and writes it into `/content/phase19_standalone`.

**Reality check:** this starts a new model from random weights and does not inherit the Phase 18 benchmark results. Four hours cannot guarantee fluency, coding, search routing, Minecraft or editing. Evaluate actual outputs below. T4 lacks native BF16 and FlashAttention 2; CUDA FP16 autocast/gradient scaling and PyTorch's supported SDPA kernel are used. The phone target refers to weight size, not RAM, speed, or a complete mobile app. ONNX needs an appropriate mobile runtime.

The default 47 minutes per stage reserves overhead; this is not four 60-minute stages. Start-to-training deadline is 3h35m, with remaining time for upload/export/evaluation, but Colab/network can interrupt earlier. Each 15-minute checkpoint is synchronous and may be late for a long step/upload; failures stop training. The private HF model repository is created if needed. Resume fetches the highest stage/step and restores optimizer, scaler, elapsed stage time and RNG. Only load checkpoints from a trusted repository (PyTorch pickle). Session memory and cache are not persisted; rebuild the same cache or supply the same data on resume. You must have HF account/repository write access.

Datasets may be gated, removed, mislabeled or unavailable; metadata is not legal advice. The notebook refuses unknown/non-permissive dataset licenses and does not fake missing real data. If a group is empty, add a local UTF-8 text file with cleared usage rights as directed below. Synthetic examples are explicitly labeled; Minecraft isn't executed, DDG requires network, grammar and code rewards are very weak proxies. Never execute model-generated code or Minecraft calls without an actual sandbox/authorization.


In [ ]:
import os, time, sys
SESSION_STARTED = time.time()
SESSION_DEADLINE = SESSION_STARTED + 3*3600 + 35*60
print('Session budget starts now. Check GPU:')
!nvidia-smi


## 1. Install dependencies
Colab supplies PyTorch. Package installation and model export use network and can take several minutes.


In [ ]:
%pip -q install "datasets>=3,<5" "huggingface_hub>=0.25,<2" "ddgs>=9,<10" "onnx>=1.16,<2" tqdm


## 2. Unpack the embedded, auditable source
The following cells write the same Python modules that accompany the notebook in the repository.


In [ ]:
from pathlib import Path
ROOT = Path('/content/phase19_standalone')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))


In [ ]:
Path('phase19').mkdir(exist_ok=True)
Path('phase19/__init__.py').write_text('"""Portable Phase 19 Colab experiment."""\n', encoding='utf8')


In [ ]:
Path('phase19').mkdir(exist_ok=True)
Path('phase19/dataset_discovery.py').write_text('"""HF discovery and bounded streaming. Only verified permissive licenses are used."""\nimport json\nimport os\nimport re\n\nQUERIES = [\'pt-br\', \'portuguese\', \'brazilian\', \'code\', \'programming\', \'python\',\n           \'javascript\', \'minecraft\', \'game commands\', \'openassistant\', \'tool\', \'agent\']\n# These are candidates, not a license or availability guarantee. bert-base is a\n# model, not a dataset. en2ptbridge must pass the exact same dataset checks.\nCANDIDATES = {\n    \'pt\': [\'maritaca-ai/en2ptbridge\', \'Helsinki-NLP/opus-100\', \'uonlp/CulturaX\'],\n    \'en\': [\'Helsinki-NLP/opus-100\', \'HuggingFaceFW/fineweb-edu\'],\n    \'code\': [\'m-a-p/CodeFeedback-Filtered-Instruction\', \'HuggingFaceH4/CodeAlpaca_20K\'],\n    \'react\': [\'OpenAssistant/oasst1\'],\n}\nALLOWED = {\'apache-2.0\', \'mit\', \'cc0-1.0\', \'cc-by-4.0\', \'cc-by-3.0\',\n           \'bsd-3-clause\', \'bsd-2-clause\', \'isc\', \'unlicense\'}\n\ndef license_of(info):\n    card = getattr(info, \'card_data\', None) or {}\n    license_ = card.get(\'license\') if hasattr(card, \'get\') else getattr(card, \'license\', None)\n    return str(license_ or \'\').lower()\n\ndef discover(limit=5):\n    from huggingface_hub import HfApi\n    api = HfApi()\n    reports = []\n    print(\'Note: pierreguillou/bert-base-cased-pt-br is a model, not a corpus dataset.\')\n    for q in QUERIES:\n        try:\n            rows = list(api.list_datasets(search=q, limit=limit, full=True))\n            for row in rows:\n                lic = license_of(row)\n                reports.append({\'query\': q, \'name\': row.id, \'license\': lic or \'unknown\',\n                    \'language\': (getattr(row, \'card_data\', None) or {}).get(\'language\', \'unknown\'),\n                    \'size_bytes\': getattr(row, \'size_in_bytes\', None),\n                    \'accepted_license\': lic in ALLOWED})\n        except Exception as exc:\n            print(\'Search failed:\', q, type(exc).__name__, str(exc)[:200])\n    for row in reports:\n        print(json.dumps(row, ensure_ascii=False, default=str))\n    return reports\n\ndef extract(row, kind):\n    if kind == \'code\':\n        q = row.get(\'instruction\') or row.get(\'query\') or row.get(\'prompt\') or row.get(\'problem\')\n        a = row.get(\'output\') or row.get(\'response\') or row.get(\'answer\') or row.get(\'completion\')\n        if isinstance(a, list):\n            a = \'\\n\'.join(str(x.get(\'content\', \'\')) for x in a if isinstance(x, dict))\n        if not isinstance(q, str) or not isinstance(a, str): return None\n        if not re.search(r\'python|javascript|\\bjs\\b|\\bdef\\b|\\bfunction\\b\', q+\' \'+a, re.I): return None\n        return \'Question: \'+q+\'\\nCode: \'+a\n    if kind == \'react\':\n        # OASST messages are not tool trajectories; do not mislabel them as such.\n        return None\n    for key in (\'text\', \'content\', \'sentence\', \'translation\'):\n        v = row.get(key)\n        if isinstance(v, dict): v = v.get(\'pt\') or v.get(\'pt-br\') or v.get(\'en\')\n        if isinstance(v, str) and len(v.strip()) >= 30: return v\n    return None\n\ndef collect(cache_dir=\'data_cache\', max_rows=4000, max_bytes=8_000_000):\n    """Stream bounded samples from explicitly licensed candidates, then cache JSON.\n\n    Dataset streams can fail or require gated access; absent groups remain empty.\n    No restricted or unknown license is silently accepted.\n    """\n    from datasets import load_dataset\n    from huggingface_hub import HfApi\n    os.makedirs(cache_dir, exist_ok=True)\n    api = HfApi(); result = {k: [] for k in CANDIDATES}; manifest = []\n    for kind, ids in CANDIDATES.items():\n        for name in ids:\n            if len(result[kind]) >= max_rows: break\n            record = {\'name\': name, \'kind\': kind}\n            try:\n                info = api.dataset_info(name)\n                lic = license_of(info)\n                record.update(license=lic or \'unknown\', language=(getattr(info, \'card_data\', None) or {}).get(\'language\', \'unknown\'), size_bytes=getattr(info, \'size_in_bytes\', None))\n                if lic not in ALLOWED:\n                    record[\'status\'] = \'skipped: unverified or non-permissive license\'\n                    continue\n                if kind == \'react\':\n                    record[\'status\'] = \'skipped: chat messages are not ReAct trajectories\'\n                    continue\n                # CulturaX has language configs; explicitly request Portuguese.\n                config = \'pt\' if name == \'uonlp/CulturaX\' else (\'en-pt\' if name == \'Helsinki-NLP/opus-100\' else None)\n                stream = load_dataset(name, config, split=\'train\', streaming=True)\n                seen_bytes = sum(len(x.encode(\'utf-8\')) for x in result[kind])\n                rows_seen = 0\n                for row in stream:\n                    rows_seen += 1\n                    text = (row.get(\'translation\', {}).get(\'pt\' if kind == \'pt\' else \'en\') if name == \'Helsinki-NLP/opus-100\' else extract(row, kind))\n                    if text:\n                        text = text[:4096]\n                        n = len(text.encode(\'utf-8\'))\n                        if seen_bytes + n > max_bytes: break\n                        result[kind].append(text); seen_bytes += n\n                    if len(result[kind]) >= max_rows or rows_seen >= max_rows * 10: break\n                record.update(status=\'streamed\', rows=len(result[kind]), cached_bytes=seen_bytes)\n            except Exception as exc:\n                record[\'status\'] = \'skipped: \' + type(exc).__name__ + \': \' + str(exc)[:200]\n            finally:\n                manifest.append(record); print(json.dumps(record, ensure_ascii=False, default=str))\n    with open(os.path.join(cache_dir, \'samples.json\'), \'w\', encoding=\'utf8\') as f:\n        json.dump(result, f, ensure_ascii=False)\n    with open(os.path.join(cache_dir, \'manifest.json\'), \'w\', encoding=\'utf8\') as f:\n        json.dump(manifest, f, ensure_ascii=False, indent=2)\n    return result\n', encoding='utf8')


In [ ]:
Path('phase19').mkdir(exist_ok=True)
Path('phase19/preprocess.py').write_text('"""Byte-level examples, bounded mixtures, deterministic synthetic data."""\nimport random\nimport re\nimport torch\n\nBOS, EOS, PAD = 256, 257, 258\n\ndef encode(text, max_seq_len=1024):\n    # Reserve EOS and BOS; no hidden tokenizer or Unicode codepoint conversion.\n    return [BOS] + list(text.encode(\'utf8\')[:max_seq_len - 2]) + [EOS]\n\ndef batch(texts, max_seq_len=1024):\n    rows = [encode(t, max_seq_len) for t in texts]\n    result = torch.full((len(rows), max(map(len, rows))), PAD, dtype=torch.long)\n    for i, row in enumerate(rows): result[i, :len(row)] = torch.tensor(row)\n    return result\n\ndef synthetic():\n    minecraft, search, react, edit = [], [], [], []\n    for x in range(-10, 11):\n        minecraft.append(f\'Request: Build a house at {x} 64 0\\nAction: minecraft.place_block("stone", {x}, 64, 0)\\nObservation: simulated placement only\\nAnswer: Primeiro bloco posicionado.\')\n    for query in [\'clima em São Paulo\', \'Python documentation\', \'notícias do Brasil\', \'weather in São Paulo\']:\n        search.append(f\'Request: {query}\\nThought: Não tenho certeza; vou pesquisar.\\nAction: search_ddg("{query}")\\nObservation: [exemplo fictício; consulte fontes atuais]\\nAnswer: Confira os resultados atuais antes de concluir.\')\n        react.append(f\'Request: {query}\\nThought: Preciso consultar uma fonte.\\nAction: search_ddg("{query}")\\nObservation: busca não executada neste exemplo\\nAnswer: Não posso verificar sem resultados reais.\')\n    for wrong, right in [(\'ola\',\'olá\'), (\'voce\',\'você\'), (\'bom diaa\',\'bom dia\'), (\'casaa\',\'casa\')]:\n        # Edit offsets and replacement bytes use UTF-8 byte indices, not character indices.\n        old, new = wrong.encode(), right.encode()\n        start = 0\n        while start < min(len(old), len(new)) and old[start] == new[start]: start += 1\n        # General edits support replacement of a byte range with arbitrary UTF-8 bytes.\n        edit.append(f\'Request: Corrija: {wrong}\\nDraft: {wrong}\\n[EDIT pos={start} delete={len(old)-start} bytes={new[start:].hex()}]\\nAnswer: {right}\')\n    return {\'minecraft\':minecraft, \'search\':search, \'react\':react, \'edit\':edit}\n\ndef apply_edits(draft, commands):\n    """Strictly apply one or more byte-range replacements; reject malformed UTF-8."""\n    pattern = re.compile(r\'\\[EDIT pos=(\\d+) delete=(\\d+) bytes=([0-9a-f]*)\\]\')\n    data = draft.encode(\'utf8\')\n    matches = list(pattern.finditer(commands))\n    if not matches or \'\'.join(m.group(0) for m in matches) != commands.strip(): raise ValueError(\'invalid edit syntax\')\n    for match in matches:\n        pos, delete = int(match[1]), int(match[2]); raw = bytes.fromhex(match[3])\n        if pos > len(data) or pos + delete > len(data): raise ValueError(\'out of range\')\n        data = data[:pos] + raw + data[pos+delete:]\n    return data.decode(\'utf8\')\n\ndef sample_mix(groups, weights, count, rng=random):\n    missing = [k for k, w in weights.items() if w and not groups.get(k)]\n    if missing: raise ValueError(\'Missing required data groups: \' + \', \'.join(missing))\n    keys = list(weights); probs = list(weights.values())\n    return [rng.choice(groups[k]) for k in rng.choices(keys, weights=probs, k=count)]\n', encoding='utf8')


In [ ]:
Path('phase19').mkdir(exist_ok=True)
Path('phase19/hf_upload.py').write_text('"""Resume snapshots: immutable paths and full optimizer/training state."""\nimport os\nimport re\nfrom pathlib import Path\nimport torch\n\nPATTERN = re.compile(r\'^checkpoints/stage_(\\d+)_step_(\\d+)/checkpoint\\.pt$\')\n\ndef save_checkpoint(state, stage, step, root=\'checkpoints\'):\n    folder = Path(root) / f\'stage_{stage}_step_{step}\'\n    folder.mkdir(parents=True, exist_ok=True)\n    tmp = folder / \'checkpoint.tmp\'\n    torch.save(state, tmp)\n    os.replace(tmp, folder / \'checkpoint.pt\')\n    return folder / \'checkpoint.pt\'\n\ndef upload_checkpoint(state, stage, step, hf_token, repo_id, root=\'checkpoints\', api=None):\n    from huggingface_hub import HfApi\n    path = save_checkpoint(state, stage, step, root)\n    api = api or HfApi(token=hf_token)\n    api.create_repo(repo_id=repo_id, repo_type=\'model\', private=True, exist_ok=True, token=hf_token)\n    api.upload_file(path_or_fileobj=str(path), path_in_repo=f\'checkpoints/stage_{stage}_step_{step}/checkpoint.pt\',\n                    repo_id=repo_id, repo_type=\'model\', token=hf_token,\n                    commit_message=f\'Stage {stage}, step {step}\')\n    return path\n\ndef latest_checkpoint(repo_id, hf_token, api=None):\n    from huggingface_hub import HfApi, hf_hub_download\n    api = api or HfApi(token=hf_token)\n    try:\n        names = api.list_repo_files(repo_id=repo_id, repo_type=\'model\', token=hf_token)\n    except Exception as exc:\n        from huggingface_hub.errors import RepositoryNotFoundError\n        if isinstance(exc, RepositoryNotFoundError): return None\n        raise\n    found = [(int(m[1]), int(m[2]), n) for n in names if (m := PATTERN.fullmatch(n))]\n    if not found: return None\n    name = max(found)[2]\n    return hf_hub_download(repo_id=repo_id, filename=name, repo_type=\'model\', token=hf_token)\n\ndef load_checkpoint(path):\n    # ONLY load checkpoints you trust: pickle can execute arbitrary code.\n    return torch.load(path, map_location=\'cpu\', weights_only=False)\n', encoding='utf8')


In [ ]:
Path('phase19').mkdir(exist_ok=True)
Path('phase19/phone_model.py').write_text('"""Small decoder-only UTF-8 byte model; not a Phase 18 checkpoint architecture."""\nfrom dataclasses import dataclass, asdict\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom torch.utils.checkpoint import checkpoint\n\n@dataclass\nclass Config:\n    vocab_size: int = 268\n    d_model: int = 512\n    n_layers: int = 12\n    n_heads: int = 8\n    d_ff: int = 2048\n    max_seq_len: int = 1024\n    dropout: float = 0.0\n    gradient_checkpointing: bool = True\n\nclass Block(nn.Module):\n    def __init__(self, cfg):\n        super().__init__(); d = cfg.d_model\n        self.heads = cfg.n_heads; self.dim = d // cfg.n_heads\n        self.ln1 = nn.LayerNorm(d); self.qkv = nn.Linear(d, 3*d); self.proj = nn.Linear(d, d)\n        self.ln2 = nn.LayerNorm(d); self.ff = nn.Sequential(nn.Linear(d, cfg.d_ff), nn.GELU(), nn.Linear(cfg.d_ff, d))\n        self.dropout = cfg.dropout\n    def forward(self, x):\n        b, t, d = x.shape\n        q, k, v = self.qkv(self.ln1(x)).chunk(3, dim=-1)\n        q, k, v = [z.view(b, t, self.heads, self.dim).transpose(1, 2) for z in (q,k,v)]\n        if torch.onnx.is_in_onnx_export():\n            mask = torch.ones((t, t), device=x.device, dtype=torch.bool).tril()\n            scores = (q @ k.transpose(-2, -1)) * self.dim ** -0.5\n            a = torch.softmax(scores.masked_fill(~mask, float(\'-inf\')), dim=-1) @ v\n        else:\n            # PyTorch chooses a compatible fused or math SDPA kernel. FlashAttention-2\n            # generally requires Ampere or newer; the T4 falls back automatically.\n            a = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)\n        x = x + self.proj(a.transpose(1, 2).contiguous().view(b,t,d))\n        return x + self.ff(self.ln2(x))\n\nclass PhoneLM(nn.Module):\n    def __init__(self, cfg=Config()):\n        super().__init__(); self.cfg = cfg\n        assert cfg.d_model % cfg.n_heads == 0\n        self.token = nn.Embedding(cfg.vocab_size, cfg.d_model)\n        self.pos = nn.Embedding(cfg.max_seq_len, cfg.d_model)\n        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])\n        self.norm = nn.LayerNorm(cfg.d_model); self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)\n        self.head.weight = self.token.weight\n        # GPT-style initialization keeps tied-embedding logits and loss well scaled.\n        self.apply(self._init_weights)\n    @staticmethod\n    def _init_weights(module):\n        if isinstance(module, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n            if isinstance(module, nn.Linear) and module.bias is not None: nn.init.zeros_(module.bias)\n    def forward(self, ids):\n        t = ids.shape[1]\n        x = self.token(ids) + self.pos(torch.arange(t, device=ids.device))\n        for block in self.blocks:\n            x = checkpoint(block, x, use_reentrant=False) if self.training and self.cfg.gradient_checkpointing else block(x)\n        return self.head(self.norm(x))\n    def loss(self, ids):\n        logits = self(ids[:, :-1]); target = ids[:, 1:]\n        return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), target.reshape(-1), ignore_index=258)\n    def freeze(self, count):\n        for p in self.parameters(): p.requires_grad = True\n        for block in self.blocks[:count]:\n            for p in block.parameters(): p.requires_grad = False\n\n@torch.no_grad()\ndef generate(model, prompt, max_new=96, sample=False, temperature=1.0):\n    model.eval(); device = next(model.parameters()).device\n    ids = [256] + list(prompt.encode(\'utf8\'))\n    ids = ids[-(model.cfg.max_seq_len-1):]\n    output = []\n    for _ in range(min(max_new, model.cfg.max_seq_len-len(ids))):\n        logits = model(torch.tensor([ids], device=device))[0, -1].float() / temperature\n        logits[258:] = -float(\'inf\') # forbid PAD and reserved ids; EOS remains available\n        nxt = int(torch.multinomial(torch.softmax(logits, -1), 1)) if sample else int(logits.argmax())\n        if nxt == 257: break\n        ids.append(nxt); output.append(nxt)\n    return bytes(output).decode(\'utf8\', errors=\'replace\')\n\ndef export_onnx(model, path):\n    # ONNX uses the explicit mathematical causal attention path; no external weights\n    # at this model size. Export CPU FP32 for compatibility; FP16 .pt is smaller.\n    model = model.to(\'cpu\').float().eval()\n    torch.onnx.export(model, (torch.tensor([[256, 79, 108, 195, 161]], dtype=torch.long),), path,\n                      input_names=[\'input_ids\'], output_names=[\'logits\'],\n                      dynamic_axes={\'input_ids\': {0:\'batch\', 1:\'sequence\'}, \'logits\': {0:\'batch\', 1:\'sequence\'}},\n                      opset_version=17, do_constant_folding=True)\n', encoding='utf8')


In [ ]:
Path('phase19').mkdir(exist_ok=True)
Path('phase19/tools.py').write_text('"""Host-side tools. No generated Python or Minecraft calls are executed here."""\nimport ast\nimport re\nfrom phase19.preprocess import apply_edits\n\ndef search_ddg(query):\n    """Return at most three live result snippets as UTF-8 bytes; network required."""\n    from ddgs import DDGS\n    results = list(DDGS().text(query, max_results=3))\n    return \'\\n\'.join(f"{r.get(\'title\', \'\')} | {r.get(\'href\', \'\')} | {r.get(\'body\', \'\')}" for r in results).encode(\'utf8\')\n\ndef minecraft_valid(text):\n    # Syntax check only, not execution against a real Minecraft server.\n    return bool(re.search(r\'minecraft\\.place_block\\("[a-z_]+",\\s*-?\\d+,\\s*-?\\d+,\\s*-?\\d+\\)\', text))\n\ndef grammar_score(text):\n    # Toy heuristic, not a Portuguese grammar checker.\n    return bool(re.search(r\'\\b(você|olá|obrigado|estou)\\b\', text.lower())) and not bool(re.search(r\'\\b(voce|ola|casaa|diaa)\\b\', text.lower()))\n\ndef reward(kind, text, draft=None, target=None):\n    if kind == \'code\':\n        source = text.strip().removeprefix(\'```python\').removesuffix(\'```\').strip()\n        try:\n            tree = ast.parse(source)\n            return float(any(isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) for n in ast.walk(tree)))\n        except (SyntaxError, ValueError): return 0.0\n    if kind == \'minecraft\': return float(minecraft_valid(text))\n    if kind == \'pt\': return .5 * grammar_score(text)\n    if kind == \'edit\' and draft is not None:\n        match = re.search(r\'\\[EDIT pos=\\d+ delete=\\d+ bytes=[0-9a-f]*\\]\', text)\n        if not match: return 0.0\n        try:\n            edited = apply_edits(draft, match.group())\n            # Exact target is a proxy for reducing errors, not an independent evaluator.\n            return .5 * (edited == target and edited != draft)\n        except (ValueError, UnicodeError): return 0.0\n    return 0.0\n\nclass Memory:\n    """Explicit host-managed session memory, not learned or persistent."""\n    def __init__(self): self.items = {}\n    def remember(self, key, value): self.items[key] = value\n    def recall(self, key): return self.items.get(key)\n    def prefix(self): return \'\\n\'.join(f\'{k}: {v}\' for k,v in self.items.items())\n\ndef run_search_action(generated, search=search_ddg):\n    """Run only an explicit, single literal search_ddg call, never eval arbitrary code."""\n    line = next((x[len(\'Action: \'):].strip() for x in generated.splitlines() if x.startswith(\'Action: search_ddg(\')), None)\n    if line is None: return None\n    try:\n        node = ast.parse(line, mode=\'eval\').body\n        if not (isinstance(node, ast.Call) and isinstance(node.func, ast.Name)\n                and node.func.id == \'search_ddg\' and len(node.args) == 1 and not node.keywords\n                and isinstance(node.args[0], ast.Constant) and isinstance(node.args[0].value, str)\n                and 0 < len(node.args[0].value) <= 200):\n            return None\n        return search(node.args[0].value)\n    except (SyntaxError, ValueError): return None\n\ndef respond(model, request, memory=None, use_search=False, max_new=96):\n    from phase19.phone_model import generate\n    context = (\'Memory:\\n\' + memory.prefix() + \'\\n\') if memory and memory.items else \'\'\n    prompt = context + \'Request: \' + request + \'\\n\'\n    output = generate(model, prompt, max_new=max_new)\n    observation = run_search_action(output) if use_search else None\n    if observation is not None:\n        output += \'\\nObservation: \' + observation.decode(\'utf8\', \'replace\') + \'\\nAnswer: \'\n        output += generate(model, prompt + output, max_new=max_new)\n    # The caller chooses whether to apply edits; no implicit manipulation of output.\n    return output\n', encoding='utf8')


In [ ]:
Path('phase19').mkdir(exist_ok=True)
Path('phase19/train.py').write_text('"""Wall-clock bounded four-stage training with recoverable HF snapshots."""\nimport math\nimport random\nimport time\nfrom dataclasses import asdict\nfrom pathlib import Path\nimport torch\nimport torch.nn.functional as F\nfrom tqdm.auto import tqdm\nfrom phase19.preprocess import batch\nfrom phase19.tools import reward\nfrom phase19.hf_upload import upload_checkpoint\n\nMIXES = [dict(pt=50, en=30, code=20), dict(code=50, minecraft=30, pt=20),\n         dict(react=40, search=30, code=30), dict(edit=50, pt=20, code=15, minecraft=15)]\nFREEZE = [0, 6, 8, 8]\nRL_CASES = [(\'code\', \'Request: Write a Python function to sort a list\\nCode: \', None, None),\n            (\'minecraft\', \'Request: Build a house at 0 64 0\\nAction: \', None, None),\n            (\'pt\', \'Request: Olá, como você está?\\nAnswer: \', None, None),\n            (\'edit\', \'Request: Corrija: voce\\nDraft: voce\\n\', \'voce\', \'você\')]\n\ndef draw(groups, weights, count):\n    keys = list(weights)\n    return [random.choice(groups[k]) for k in random.choices(keys, weights=list(weights.values()), k=count)]\n\ndef check_groups(groups):\n    missing = sorted({k for mix in MIXES for k in mix if not groups.get(k)})\n    if missing: raise ValueError(\'Missing training groups: \' + \', \'.join(missing) + \'. Supply licensed UTF-8 text files; do not silently substitute synthetic data.\')\n\ndef snapshot(model, opt, scaler, stage, step, elapsed, cfg):\n    return dict(version=1, model=model.state_dict(), optimizer=opt.state_dict(), scaler=scaler.state_dict(),\n                stage=stage, step=step, elapsed=elapsed, config=asdict(cfg), random=random.getstate(),\n                torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)\n\ndef policy_loss(model, device, max_new=48):\n    """REINFORCE: sample token IDs, then replay them to preserve policy gradients."""\n    kind, prompt, draft, target = random.choice(RL_CASES)\n    prefix = [256] + list(prompt.encode(\'utf8\')); ids = prefix[:]\n    model.eval()\n    with torch.no_grad():\n        for _ in range(min(max_new, model.cfg.max_seq_len-len(ids))):\n            logits = model(torch.tensor([ids], device=device))[0, -1].float()\n            logits[258:] = -float(\'inf\')\n            token = int(torch.multinomial(torch.softmax(logits, -1), 1)); ids.append(token)\n            if token == 257: break\n    output = bytes(x for x in ids[len(prefix):] if x < 256).decode(\'utf8\', \'replace\')\n    score = reward(kind, output, draft, target)\n    model.train()\n    if score == 0 or len(ids) == len(prefix): return None, score\n    inp = torch.tensor([ids[:-1]], device=device)\n    logits = model(inp)[0, len(prefix)-1:].float()\n    logits = torch.cat([logits[:, :258], torch.full_like(logits[:, 258:], -float(\'inf\'))], dim=-1)\n    labels = torch.tensor(ids[len(prefix):], device=device)\n    return -score * F.log_softmax(logits, dim=-1).gather(1, labels[:,None]).mean(), score\n\ndef train(model, groups, token, repo_id, session_deadline, stage_seconds=47*60,\n          upload_seconds=15*60, micro_batch=1, accumulation=32, resume=None, lr=3e-4):\n    check_groups(groups)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    model.to(device)\n    if resume:\n        if resume[\'config\'] != asdict(model.cfg): raise ValueError(\'Checkpoint architecture mismatch\')\n        model.load_state_dict(resume[\'model\']); random.setstate(resume[\'random\']); torch.set_rng_state(resume[\'torch_rng\'])\n        if torch.cuda.is_available() and resume[\'cuda_rng\'] is not None: torch.cuda.set_rng_state_all(resume[\'cuda_rng\'])\n    start_stage = resume[\'stage\'] if resume else 0\n    step = resume[\'step\'] if resume else 0\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=device.type == \'cuda\')\n    for stage in range(start_stage, 4):\n        model.freeze(min(FREEZE[stage], len(model.blocks)))\n        opt = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=lr, weight_decay=.01)\n        elapsed = resume[\'elapsed\'] if resume and stage == start_stage else 0.0\n        if resume and stage == start_stage:\n            opt.load_state_dict(resume[\'optimizer\']); scaler.load_state_dict(resume[\'scaler\'])\n        last_upload = time.monotonic()\n        bar = tqdm(total=stage_seconds, initial=min(elapsed, stage_seconds), desc=f\'Stage {stage+1}\', unit=\'s\')\n        tick = time.monotonic(); model.train()\n        while elapsed < stage_seconds and time.time() < session_deadline:\n            opt.zero_grad(set_to_none=True)\n            frac = min(1., elapsed/stage_seconds)\n            for param in opt.param_groups: param[\'lr\'] = lr * .5 * (1 + math.cos(math.pi * frac))\n            losses = []\n            for _ in range(accumulation):\n                ids = batch(draw(groups, MIXES[stage], micro_batch), model.cfg.max_seq_len).to(device)\n                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == \'cuda\'):\n                    loss = model.loss(ids)/accumulation\n                scaler.scale(loss).backward(); losses.append(float(loss.detach())*accumulation)\n            if stage == 3 and step % 8 == 0:\n                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == \'cuda\'):\n                    rl_loss, score = policy_loss(model, device)\n                if rl_loss is not None: scaler.scale(rl_loss * .1).backward()\n            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            scaler.step(opt); scaler.update(); step += 1\n            now = time.monotonic(); delta = now-tick; tick = now; elapsed += delta\n            bar.update(min(delta, max(0, stage_seconds-bar.n))); bar.set_postfix(loss=f\'{sum(losses)/len(losses):.3f}\', step=step)\n            if now-last_upload >= upload_seconds:\n                upload_checkpoint(snapshot(model, opt, scaler, stage, step, elapsed, model.cfg), stage, step, token, repo_id)\n                last_upload = time.monotonic()  # upload time counts against the budget\n        bar.close()\n        # Upload current stage even on wall-clock stop. At resume, an exhausted\n        # stage immediately advances after restoring its optimizer.\n        upload_checkpoint(snapshot(model, opt, scaler, stage, step, elapsed, model.cfg), stage, step, token, repo_id)\n        resume = None\n        if time.time() >= session_deadline:\n            print(\'Stopped at session deadline; resume from the latest uploaded checkpoint.\')\n            return False\n    return True\n', encoding='utf8')


In [ ]:
Path('phase19').mkdir(exist_ok=True)
Path('phase19/evaluate.py').write_text('"""Free-running smoke probes. These are diagnostics, not proof of capability."""\nimport ast\nimport re\nfrom phase19.phone_model import generate\nfrom phase19.tools import Memory, minecraft_valid, reward\n\nCASES = [\n (\'pt-br\', \'Request: Olá, como você está?\\nAnswer: \'),\n (\'code\', \'Request: Write a Python function to sort a list\\nCode: \'),\n (\'minecraft\', \'Request: Build a house at 0 64 0\\nAction: \'),\n (\'search\', \'Request: What is the weather in São Paulo?\\nAction: \'),\n (\'edit\', \'Request: Corrija: voce\\nDraft: voce\\n\'),\n (\'memory\', \'Memory:\\nMeu animal favorito: capivara\\nRequest: Qual é meu animal favorito?\\nAnswer: \'),\n]\n\ndef evaluate(model, max_new=128):\n    rows = []\n    for kind, prompt in CASES:\n        output = generate(model, prompt, max_new=max_new)\n        if kind == \'pt-br\': ok = bool(re.search(r\'\\b(estou|bem|olá|obrigad[oa]|você)\\b\', output.lower()))\n        elif kind == \'code\': ok = bool(reward(\'code\', output))\n        elif kind == \'minecraft\': ok = minecraft_valid(output)\n        elif kind == \'search\': ok = \'search_ddg(\' in output\n        elif kind == \'edit\': ok = bool(reward(\'edit\', output, \'voce\', \'você\'))\n        else: ok = \'capivara\' in output.lower()\n        rows.append({\'case\': kind, \'pass\': bool(ok), \'output\': output})\n    return rows\n', encoding='utf8')


## 3. Credentials and experiment settings
Use a **write-scoped** HF token entered privately via the input prompt; do not paste it into code or publish a saved notebook with secrets. Set `repo_id` to your own account/repository. Resume trusts uploaded pickle data. Batch size 1 × accumulation 32 defaults to effective 32; increase micro batch only if T4 VRAM allows. Stage lengths may be changed for experiments, but four 60-minute stages do not fit a 4-hour session with overhead.


In [ ]:
from getpass import getpass
import torch, random
from phase19.phone_model import Config, PhoneLM
from phase19.hf_upload import latest_checkpoint, load_checkpoint
HF_TOKEN = getpass('Hugging Face write token: ')
REPO_ID = input('Private model repo (username/repo): ').strip()
if not HF_TOKEN or '/' not in REPO_ID: raise ValueError('A token and username/repo are required')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (T4 strongly recommended)')
random.seed(19); torch.manual_seed(19)
cfg = Config()
model = PhoneLM(cfg)
print('Parameters:', sum(p.numel() for p in model.parameters()), '; raw FP16 MB:', sum(p.numel() for p in model.parameters())*2/1e6)
checkpoint_path = latest_checkpoint(REPO_ID, HF_TOKEN)
resume = load_checkpoint(checkpoint_path) if checkpoint_path else None
print('Resume:', checkpoint_path or 'starting from scratch')


## 4. Discover and cache real datasets
Results include dataset IDs, language, declared license, and size where exposed. The preselected `pierreguillou/bert-base-cased-pt-br` is a **model**, not a dataset; `maritaca-ai/en2ptbridge` is inspected as a candidate, not assumed to exist or be useful. OASST chat messages are not falsely relabeled as tool trajectories. Candidate datasets are sampled via streaming and hard per-group limits; unknown or non-permissive licenses are skipped. Review `data_cache/manifest.json` and source terms yourself.


In [ ]:
from phase19.dataset_discovery import discover, collect
from phase19.preprocess import synthetic
reports = discover(limit=5)
groups = collect(max_rows=4000, max_bytes=8_000_000)
groups.update(synthetic())
print({k: len(v) for k,v in groups.items()})


### Missing real data?
If `pt`, `en`, or `code` is empty, upload or write a UTF-8 text file per group in `/content/phase19_standalone/manual_data/pt.txt`, `en.txt`, or `code.txt` and rerun the next cell. One example per line. You are responsible for permission and license compatibility; report provenance separately. The trainer refuses empty groups rather than quietly substituting synthetic rows.


In [ ]:
from phase19.train import check_groups
for kind in ('pt', 'en', 'code'):
    path = ROOT / 'manual_data' / (kind+'.txt')
    if path.exists():
        with path.open(encoding='utf8') as f:
            groups[kind].extend(line.strip()[:4096] for line in f if line.strip())
print({k: len(v) for k,v in groups.items()})
check_groups(groups)


## 5. Train and upload
Stages use 50/30/20 pt/en/code, 50/30/20 code/Minecraft/pt, and 40/30/30 ReAct/search/code. The last stage alternates supervised edits with sparse sampled REINFORCE and weak, non-executing rewards. Freeze counts are 0/6/8/8 blocks. Sequences are UTF-8 bytes truncated to 1024. If checkpoint upload fails, training raises; the local snapshot remains in `checkpoints/` but is lost if the runtime disappears.


In [ ]:
from phase19.train import train
completed = train(model, groups, HF_TOKEN, REPO_ID, SESSION_DEADLINE,
                  stage_seconds=47*60, upload_seconds=15*60, micro_batch=1, accumulation=32, resume=resume)
print('All four stages completed:', completed)


## 6. Evaluate actual free-running outputs
These six tiny probes are not statistically valid benchmarks, and passing heuristics do not establish correctness. Code is parsed, never executed; Minecraft is syntax-checked only. Memory is supplied in the prompt by the host, not stored inside the network. Search is checked for a generated call, not a real DDG request. Model outputs can fail every probe.


In [ ]:
from phase19.evaluate import evaluate
results = evaluate(model)
for row in results:
    print('\n', row['case'], 'PASS' if row['pass'] else 'FAIL', '\n', repr(row['output']))
print('Probes passed:', sum(x['pass'] for x in results), '/', len(results))


## 7. Export and download
FP16 weights are exported separately from large resumable optimizer checkpoints. ONNX exports FP32 for compatibility and may be ~2× larger; it is an optional runtime representation, not a verified phone app. The model is moved to CPU by export. Verify numerical and mobile-runtime behavior on your target device before deployment.


In [ ]:
import json, shutil, zipfile
from dataclasses import asdict
from phase19.phone_model import export_onnx
out = ROOT/'phone_export'; out.mkdir(exist_ok=True)
torch.save({k:v.detach().cpu().half() for k,v in model.state_dict().items()}, out/'phone_fp16.pt')
(out/'config.json').write_text(json.dumps(asdict(cfg), indent=2))
shutil.copytree(ROOT/'phase19', out/'phase19', dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__', 'build_phase19.py'))
try:
    export_onnx(model, str(out/'phone_fp32.onnx'))
    print('ONNX export succeeded:', (out/'phone_fp32.onnx').stat().st_size/1e6, 'MB')
except Exception as exc:
    print('ONNX export FAILED:', type(exc).__name__, str(exc))
# Download weights/source separately from ONNX to keep the primary artifact small.
with zipfile.ZipFile(ROOT/'phone_model_fp16.zip', 'w', compression=zipfile.ZIP_STORED) as z:
    for path in out.rglob('*'):
        if path.is_file() and path.suffix != '.onnx': z.write(path, path.relative_to(out))
print('FP16 bundle MB:', (ROOT/'phone_model_fp16.zip').stat().st_size / 1e6)
from google.colab import files
files.download(str(ROOT/'phone_model_fp16.zip'))
# Optional: uncomment to download the larger ONNX model separately.
# files.download(str(out/'phone_fp32.onnx'))
